# Kombinierte Labels: Extra Trees

**Hypothese:** Zufälligere Baumaufteilungen erfassen die stark korrelierten Spektralbänder besser als Random Forest und übertreffen dessen kombinierte CV-BAcc von 0,788.

Die Hyperparameter werden ausschließlich auf den fünf gemeinsamen Trainings-Folds ausgewählt. Der Validierungsanteil bleibt bis zu einem einzelnen, nur bei überzeugender CV ausgeführten `run()`-Lauf unberührt.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from IPython.display import display

from awp2.experiment import RunConfig, TuneConfig, run, tune
from awp2.models import make_combined_extra_trees
from awp2.preprocessing import PreprocessingConfig

COMBINED_RF_CV_BACC = 0.788
COMBINED_SVM_CV_BACC = 0.834
COMBINED_SVM_VAL_BACC = 0.845

extra_trees_preprocessing = PreprocessingConfig(scale=False)

## 1. Hyperparameter suchen

`TuneConfig` lädt den gemeinsamen Trainingsanteil und die vorgegebenen Folds. Die acht Kandidaten variieren nur die Merkmalsanzahl, Blattgröße und Baumtiefe; Anzahl der Bäume, Klassengewichte und Seed bleiben fest.

In [ ]:
tuned = tune(
    make_combined_extra_trees(),
    TuneConfig(
        name="combined_extra_trees_search",
        approach="combined",
        description="Extra Trees auf kombinierten Crop/Stage-Labels: Merkmalsanteil, Blattgröße und Tiefe vergleichen.",
        preprocessing=extra_trees_preprocessing,
        param_grid={
            "estimator__max_features": [0.3, "sqrt"],
            "estimator__min_samples_leaf": [1, 2],
            "estimator__max_depth": [None, 20],
        },
    ),
)

display(tuned.candidates.sort_values("rank"))
{
    "best_cv_score": tuned.best_score,
    "best_params": tuned.best_params,
    "tuning_run": tuned.run_id,
}

## 2. Einmal validieren

Die beste CV-Konfiguration übertrifft die Random-Forest-Referenz um mehr als zwei BAcc-Punkte. Daher wird sie genau einmal auf dem gemeinsamen, bisher unberührten Validierungsanteil bewertet.

In [ ]:
result = run(
    tuned.best_model,
    RunConfig(
        name="combined_extra_trees",
        approach="combined",
        description="Beste Extra-Trees-Konfiguration aus combined_extra_trees_search, einmal auf der Validierung bewertet.",
        preprocessing=extra_trees_preprocessing,
        tuning_run=tuned.run_id,
    ),
)

result.metrics.model_dump()

## 3. Analyse

Die Vergleichstabelle trennt CV und Validierung bewusst. Die Abbildungen zeigen die Kandidatenstreuung, die Verwechslungen und den Recall der einzelnen kombinierten Klassen.

In [ ]:
from awp2.data import combined_label
from awp2.evaluation import plot_confusion_matrices
from awp2.plots import plot_class_recall, plot_top_confusions, plot_tuning_candidates

comparison = pd.DataFrame(
    {
        "CV-BAcc kombiniert": {
            "Random Forest": COMBINED_RF_CV_BACC,
            "Extra Trees": tuned.best_score,
            "RBF-SVM": COMBINED_SVM_CV_BACC,
        },
        "Validierungs-BAcc kombiniert": {
            "Extra Trees": result.metrics.bacc_combined,
            "RBF-SVM": COMBINED_SVM_VAL_BACC,
        },
    }
)
display(comparison)

top_confusions = (
    pd.DataFrame({"Wahr": combined_label(result.y_val), "Vorhergesagt": combined_label(result.y_pred)})
    .query("Wahr != Vorhergesagt")
    .value_counts()
    .head(10)
)
display(top_confusions)

tuning_ax = plot_tuning_candidates(
    tuned.candidates,
    color_by="estimator__max_features",
    validation_score=result.metrics.bacc_combined,
)
tuning_ax.set_title("Kombinierte Labels: CV-Score von Extra Trees")
display(tuning_ax.figure)

for figure in (
    plot_confusion_matrices(result.y_val, result.y_pred),
    plot_class_recall(result.y_val, result.y_pred),
    plot_top_confusions(result.y_val, result.y_pred),
):
    display(figure)

In [ ]:
from awp2.config import MODEL_DOCS_FIGURES_DIR
from awp2.plots import save_doc_figure

save_doc_figure(tuning_ax.figure, "combined_extra_trees_tuning", MODEL_DOCS_FIGURES_DIR)

confusion_figure = plot_confusion_matrices(result.y_val, result.y_pred)
save_doc_figure(confusion_figure, "combined_extra_trees_confusion", MODEL_DOCS_FIGURES_DIR)

recall_figure = plot_class_recall(result.y_val, result.y_pred)
save_doc_figure(recall_figure, "combined_extra_trees_recall", MODEL_DOCS_FIGURES_DIR)

confusions_figure = plot_top_confusions(result.y_val, result.y_pred)
save_doc_figure(confusions_figure, "combined_extra_trees_confusions", MODEL_DOCS_FIGURES_DIR)

{
    "tuning_run": tuned.run_id,
    "evaluation_run": result.run_id,
    "invalid_combinations": result.metrics.invalid_combinations,
}